# CYS5550: Introduction to Artificial Intelligence
# Homework: Object Detection & Segmentation with YOLO

---

## Assignment Information

| Item | Details |
|------|---------|
| **Course** | CYS5550 – Introduction to Artificial Intelligence |
| **Assignment** | Weeks 7 Homework |
| **Topics** | Computer Vision, Object Detection, Instance Segmentation, Deep Learning |
| **Total Points** | 100 |
| **Estimated Time** | 3–4 hours |
| **Due Date** | 03/03/2026 11:59pm  |
| **Submission** | Submit this completed notebook (.ipynb) and all generated output images via Github |

## Overview

In this assignment you will work with **YOLOv8**, a state-of-the-art real-time object detection and instance segmentation model. You will:

1. **Detect** objects in images using a pre-trained YOLO model
2. **Segment** objects at the pixel level using YOLO's segmentation variant
3. **Analyze** detection results quantitatively (confidence, class distribution)
4. **Compare** classical CV segmentation with deep-learning segmentation on the same image
5. **Reflect** on the three eras of computer vision covered in Lectures 7–8

### Connection to Lectures

| Lecture Concept | Homework Task |
|----------------|---------------|
| Era 1 — Classical CV (edges, thresholds, contours) | Part 3: Classical baseline |
| Era 3 — Deep learning detection (YOLO, Faster R-CNN) | Parts 1–2: YOLOv8 detection & segmentation |
| Transfer learning / pre-trained models | Using COCO-pretrained YOLO |
| Comparing eras & choosing approaches | Part 3 comparison + Part 4 reflection |


## Academic Integrity

- You may use documentation, lecture notes, and the Ultralytics docs (https://docs.ultralytics.com).
- You may **not** use AI assistants to write your code or reflection answers.
- All reflection answers must be in your own words.

## Student Information

**Fill in before submission:**

In [ ]:
# ══════════════════════════════════════════════════════
# STUDENT INFORMATION — Fill in your details
# ══════════════════════════════════════════════════════

STUDENT_NAME = "YOUR NAME HERE"
STUDENT_ID   = "YOUR ID HERE"

print(f'Student: {STUDENT_NAME}')
print(f'ID:      {STUDENT_ID}')

---
## Part 0: Environment Setup (0 points — but required!)

Run the cells below to install libraries and verify your environment.

**Google Colab is recommended** (free T4 GPU). If running locally, ensure you have Python 3.10+.

In [ ]:
# ══════════════════════════════════════════════════════
# SETUP — Run this cell first
# ══════════════════════════════════════════════════════

# Uncomment the line below if packages are not installed
# !pip install ultralytics opencv-python matplotlib numpy

import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import time
import os

from ultralytics import YOLO

print('OpenCV:', cv2.__version__)
print('NumPy:', np.__version__)
print('Ultralytics YOLO ready')

import torch
if torch.cuda.is_available():
    print(f'GPU available: {torch.cuda.get_device_name(0)}')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    print('Apple MPS (Metal) available')
else:
    print('Running on CPU (slower but works fine)')

os.makedirs('hw_outputs', exist_ok=True)
print('\nSetup complete!')

### Prepare Sample Images

Run the cell below to download test images and create a synthetic bin scene.

In [ ]:
# ══════════════════════════════════════════════════════
# DOWNLOAD / PREPARE SAMPLE IMAGES
# ══════════════════════════════════════════════════════

import urllib.request

sample_urls = {
    "street.jpg": "https://ultralytics.com/images/bus.jpg",
    "office.jpg": "https://ultralytics.com/images/zidane.jpg",
}

os.makedirs('sample_images', exist_ok=True)

for filename, url in sample_urls.items():
    filepath = f'sample_images/{filename}'
    if not os.path.exists(filepath):
        print(f'Downloading {filename}...')
        urllib.request.urlretrieve(url, filepath)
    else:
        print(f'{filename} already exists')


def create_bin_scene(width=640, height=480):
    """Create a synthetic overhead bin scene with shapes (washers, bolts, nuts)."""
    img = np.zeros((height, width, 3), dtype=np.uint8)
    img[:] = (50, 50, 55)
    cv2.rectangle(img, (15, 15), (width-15, height-15), (90, 90, 95), 3)

    # Washers (circles with holes)
    for (cx, cy, r) in [(120, 120, 35), (400, 300, 30), (250, 380, 28)]:
        cv2.circle(img, (cx, cy), r, (180, 180, 185), -1)
        cv2.circle(img, (cx, cy), r // 3, (50, 50, 55), -1)

    # Bolts (lines with circle heads)
    for (cx, cy, angle) in [(300, 100, 30), (500, 200, -20), (150, 300, 70)]:
        length, thick = 60, 12
        dx = int(length * np.cos(np.radians(angle)))
        dy = int(length * np.sin(np.radians(angle)))
        cv2.line(img, (cx, cy), (cx+dx, cy+dy), (190, 190, 195), thick)
        cv2.circle(img, (cx, cy), thick, (210, 210, 215), -1)

    # Nuts (hexagons with holes)
    for (cx, cy, sz) in [(450, 400, 22), (80, 400, 20)]:
        pts = []
        for i in range(6):
            a = np.radians(60 * i + 15)
            pts.append([int(cx + sz*np.cos(a)), int(cy + sz*np.sin(a))])
        cv2.fillPoly(img, [np.array(pts)], (175, 175, 180))
        cv2.circle(img, (cx, cy), sz // 3, (50, 50, 55), -1)

    noise = np.random.normal(0, 4, img.shape).astype(np.int16)
    img = np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    return img


bin_scene = create_bin_scene()
cv2.imwrite('sample_images/bin_scene.png', bin_scene)
print('Synthetic bin scene created')

# Display all sample images
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, fname in zip(axes, ['street.jpg', 'office.jpg', 'bin_scene.png']):
    img = cv2.imread(f'sample_images/{fname}')
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(fname, fontsize=12)
    ax.axis('off')
plt.suptitle('Sample Images for This Assignment', fontsize=14)
plt.tight_layout()
plt.show()

---
# Part 1: Object Detection with YOLOv8 (30 points)

In this section you will load a **pre-trained YOLOv8 detection model** (trained on the 80-class COCO dataset) and use it to detect objects in images.

COCO classes include: person, bicycle, car, bus, truck, traffic light, dog, cat, bottle, cup, chair, laptop, cell phone, book, and 65 more.

### Task 1.1: Load Model and Run Detection (10 points)

1. Load the `yolov8n.pt` (nano) detection model.
2. Run inference on `sample_images/street.jpg`.
3. Display the annotated image with bounding boxes.
4. Print a summary of what was detected (class name, confidence, bounding box coordinates).

In [ ]:
# ══════════════════════════════════════════════════════
# TASK 1.1: Load YOLO and run detection (10 pts)
# ══════════════════════════════════════════════════════

# TODO: Load the YOLOv8 nano detection model
model_detect = None  # REPLACE with your code

# TODO: Run inference on the street image
results = None  # REPLACE with your code

# TODO: Display the annotated result image
# YOUR CODE HERE


# TODO: Print detection details
# For each detected object, print: class name, confidence, bounding box
print('\nDetection Summary:')
print(f'{"Class":<15} {"Conf":>6} {"Box (x1,y1,x2,y2)"}')
print('-' * 55)

# YOUR CODE HERE


### Task 1.2: Analyze Detection Results (10 points)

Using the detection results from Task 1.1:

1. **Count** the total number of objects detected.
2. **Create a bar chart** showing the number of detections per class.
3. **Create a histogram** of confidence scores across all detections.
4. **Filter** and print only objects with confidence ≥ 0.7.

In [ ]:
# ══════════════════════════════════════════════════════
# TASK 1.2: Analyze detection results (10 pts)
# ══════════════════════════════════════════════════════

from collections import Counter

# TODO 1: Count total detections
total_detections = 0  # REPLACE
print(f'Total objects detected: {total_detections}')


# TODO 2: Bar chart of detections per class
# Hint: collect class names into a list, count with Counter,
#        then use plt.barh(names, counts)

# YOUR CODE HERE


# TODO 3: Histogram of confidence scores
# Hint: collect all box.conf values, use plt.hist(confs, bins=10)

# YOUR CODE HERE


# TODO 4: Filter high-confidence detections (>= 0.7)
print('\nHigh-confidence detections (conf >= 0.7):')
print('-' * 50)

# YOUR CODE HERE


### Task 1.3: Detect Objects in Your Own Image (10 points)

1. **Upload** an image of your choice. It should contain at least **3 different object types** that COCO recognizes.
   - Colab: `from google.colab import files; uploaded = files.upload()`
   - Local: place the file in `sample_images/`
2. Run YOLOv8 detection on your image.
3. Display the annotated result.
4. Print all detected objects with confidence scores.
5. **Save** the annotated image to `hw_outputs/task1_3_my_image.jpg`.

In [ ]:
# ══════════════════════════════════════════════════════
# TASK 1.3: Detect objects in YOUR image (10 pts)
# ══════════════════════════════════════════════════════

# Uncomment for Colab upload:
# from google.colab import files
# uploaded = files.upload()
# my_image_path = list(uploaded.keys())[0]

my_image_path = "sample_images/YOUR_IMAGE.jpg"  # CHANGE THIS

# TODO: Run detection

# YOUR CODE HERE


# TODO: Display annotated result

# YOUR CODE HERE


# TODO: Print detections
print('Detections in my image:')
print('-' * 50)

# YOUR CODE HERE


# TODO: Save annotated image to hw_outputs/task1_3_my_image.jpg

# YOUR CODE HERE


---
# Part 2: Instance Segmentation with YOLOv8 (30 points)

Detection gives **bounding boxes** (rectangles). Instance segmentation gives **pixel masks** — the exact shape of each object.

YOLOv8 has a segmentation variant (`yolov8n-seg.pt`) that predicts both boxes AND masks in one forward pass.

**Why this matters for bin picking:** Bounding boxes overlap when parts are close together. Pixel masks give the exact boundary of each part, enabling more precise grasp planning.

### Task 2.1: Load Segmentation Model and Run Inference (10 points)

1. Load `yolov8n-seg.pt` (nano segmentation model).
2. Run inference on `sample_images/street.jpg`.
3. Display the image with segmentation masks overlaid.
4. Print the number of objects segmented and their class names.

In [ ]:
# ══════════════════════════════════════════════════════
# TASK 2.1: YOLO segmentation (10 pts)
# ══════════════════════════════════════════════════════

# TODO: Load the YOLOv8 nano SEGMENTATION model
# Hint: model_seg = YOLO('yolov8n-seg.pt')
model_seg = None  # REPLACE

# TODO: Run segmentation on street.jpg
results_seg = None  # REPLACE

# TODO: Display annotated image (with masks)
# Hint: results_seg[0].plot() draws both boxes and masks

# YOUR CODE HERE


# TODO: Print summary (how many objects, their classes)
print('Segmentation Summary:')
print('-' * 50)

# YOUR CODE HERE


### Task 2.2: Extract and Visualize Individual Masks (10 points)

1. Extract the binary mask for **each** detected object.
2. Display a grid: original image + up to 6 individual object masks.
3. For each mask, compute and print **pixel area** and **% of image** the object covers.

**Hint:** `results_seg[0].masks.data` is a tensor of shape `(N, H, W)`. Each slice `[i]` is a binary mask for one object.

In [ ]:
# ══════════════════════════════════════════════════════
# TASK 2.2: Extract individual masks (10 pts)
# ══════════════════════════════════════════════════════

# TODO: Extract masks and detection info
# Hint:
#   masks_tensor = results_seg[0].masks.data   # (N, H, W)
#   masks_np = masks_tensor.cpu().numpy()
#   boxes = results_seg[0].boxes

# YOUR CODE HERE — extract masks and boxes


# TODO: Display grid (original + up to 6 individual masks)
# For each mask subplot:
#   - Show mask with cmap='gray' or a colormap
#   - Title = class name + confidence
#   - pixel_area = np.sum(mask > 0.5)
#   - coverage = pixel_area / (H * W) * 100

# YOUR CODE HERE — create subplot grid


# TODO: Print mask statistics table
print('\nMask Statistics:')
print(f'{"Object":<25} {"Pixel Area":>12} {"Coverage %":>12}')
print('-' * 50)

# YOUR CODE HERE


### Task 2.3: Isolate a Single Object Using Its Mask (10 points)

Pick **one** detected object. Use its mask to:

1. **Isolate** the object (set all non-object pixels to black).
2. **Crop** the image to the object's bounding box.
3. Display three panels: (a) original with object highlighted, (b) isolated on black, (c) cropped.
4. **Save** the cropped result to `hw_outputs/task2_3_isolated.jpg`.

_This is exactly what a bin-picking system does: segment individual parts before grasp planning._

In [ ]:
# ══════════════════════════════════════════════════════
# TASK 2.3: Isolate a single object (10 pts)
# ══════════════════════════════════════════════════════

OBJECT_INDEX = 0  # Change this to select a different object

original = cv2.imread('sample_images/street.jpg')
original_rgb = cv2.cvtColor(original, cv2.COLOR_BGR2RGB)
H_orig, W_orig = original_rgb.shape[:2]

# TODO: Get mask for chosen object and resize to match original image
# YOUR CODE HERE — get and resize mask


# TODO: Create isolated image (original where mask=1, black elsewhere)
# YOUR CODE HERE


# TODO: Crop to bounding box
# YOUR CODE HERE


# TODO: Display 3-panel figure
# (a) Original with highlight  (b) Isolated on black  (c) Cropped

# YOUR CODE HERE


# TODO: Save cropped isolated object
# YOUR CODE HERE


---
# Part 3: Classical CV vs. Deep Learning Segmentation (25 points)

In Lecture 7 we learned that Era 1 (Classical CV) used thresholding, edge detection, and contour analysis to segment objects. Now you will compare that approach directly with YOLO on the **same** image.

We use the synthetic bin scene — it is designed to work with both classical and deep learning methods.

### Task 3.1: Classical Segmentation Pipeline (15 points)

Build a classical CV pipeline to segment parts in the bin image. Your pipeline must include:

1. **Preprocessing** — Grayscale conversion + Gaussian or bilateral blur
2. **Thresholding** — Otsu's method to create a binary mask
3. **Morphological cleanup** — Opening or closing to remove noise
4. **Contour detection** — `cv2.findContours`
5. **Visualization** — Draw detected contours (each a different color) on the original
6. **Statistics** — Print number of objects, and for each: area, perimeter, circularity, bounding box

In [ ]:
# ══════════════════════════════════════════════════════
# TASK 3.1: Classical CV segmentation pipeline (15 pts)
# ══════════════════════════════════════════════════════

bin_img = cv2.imread('sample_images/bin_scene.png')
bin_rgb = cv2.cvtColor(bin_img, cv2.COLOR_BGR2RGB)

# STEP 1: Convert to grayscale + blur
# YOUR CODE HERE


# STEP 2: Otsu's thresholding
# YOUR CODE HERE


# STEP 3: Morphological cleanup
# YOUR CODE HERE


# STEP 4: Find contours
# YOUR CODE HERE


# STEP 5: Draw contours — each a different color
# Hint: color = tuple(np.random.randint(50, 255, 3).tolist())

classical_result = bin_rgb.copy()

# YOUR CODE HERE


# Display
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].imshow(bin_rgb)
axes[0].set_title('Original Bin')
# axes[1].imshow(binary_or_cleaned, cmap='gray')  # ← uncomment & adjust
# axes[1].set_title('Binary Mask')
axes[2].imshow(classical_result)
axes[2].set_title('Detected Contours')
for ax in axes: ax.axis('off')
plt.tight_layout()
plt.show()


# STEP 6: Print statistics (filter contours with area > 100)
print('Classical CV Segmentation Results:')
print(f'{"#":<4} {"Area":>8} {"Perim":>8} {"Circ":>6} {"BBox (x,y,w,h)"}')
print('-' * 55)

# YOUR CODE HERE — loop through contours, compute stats, print
# area = cv2.contourArea(cnt)
# perimeter = cv2.arcLength(cnt, True)
# circularity = 4*np.pi*area / (perimeter**2) if perimeter > 0 else 0
# x, y, w, h = cv2.boundingRect(cnt)

# YOUR CODE HERE


### Task 3.2: Side-by-Side Comparison (10 points)

1. Run YOLO segmentation on `sample_images/bin_scene.png`.
2. Create a **3-panel figure**: (a) Original, (b) Classical CV result, (c) YOLO result.
3. Answer the comparison questions in the cell that follows the figure.

_Note: YOLO was trained on COCO (real-world photos). It will likely not recognize synthetic shapes — this is an important observation to discuss!_

In [ ]:
# ══════════════════════════════════════════════════════
# TASK 3.2: Side-by-side comparison (10 pts)
# ══════════════════════════════════════════════════════

# Run YOLO segmentation on the bin scene
bin_results_seg = model_seg('sample_images/bin_scene.png', verbose=False)

yolo_annotated = bin_results_seg[0].plot()
yolo_annotated_rgb = cv2.cvtColor(yolo_annotated, cv2.COLOR_BGR2RGB)

# TODO: Create 3-panel figure
# (a) Original bin image
# (b) Classical CV result (classical_result from Task 3.1)
# (c) YOLO result (yolo_annotated_rgb)

# YOUR CODE HERE


# Save comparison figure
# plt.savefig('hw_outputs/task3_2_comparison.png', dpi=150, bbox_inches='tight')

# YOUR CODE HERE


In [ ]:
# ══════════════════════════════════════════════════════
# TASK 3.2 (continued): Written comparison
# ══════════════════════════════════════════════════════

# Fill in your observations below.

comparison = {
    "classical_objects_found": 0,           # Fill in
    "yolo_objects_found": 0,                # Fill in
    "which_had_tighter_boundaries": "___",  # "classical" or "yolo" or "neither"
    "explanation": """
    REPLACE THIS with 3-5 sentences explaining:
    - Why did the two methods produce different results?
    - Why might YOLO struggle on the synthetic bin image?
    - In what scenario would each method be the better choice?
    """
}

print('Comparison Results:')
print(f'  Classical CV found: {comparison["classical_objects_found"]} objects')
print(f'  YOLO found:         {comparison["yolo_objects_found"]} objects')
print(f'  Tighter boundaries: {comparison["which_had_tighter_boundaries"]}')
print(f'\nExplanation:\n{comparison["explanation"]}')

---
# Part 4: Reflection Questions (15 points)

Answer each question in **4–6 complete sentences**. Demonstrate understanding of Lectures 7 and 8.

In [ ]:
# ══════════════════════════════════════════════════════
# REFLECTION 1 (5 pts)
# ══════════════════════════════════════════════════════
#
# QUESTION: Explain why a real-world bin-picking factory
# might use BOTH classical CV and deep learning in the same
# system rather than relying on just one. Give a specific
# example of what each approach would handle.
#
# ─────────────────────────────────────────────────────

REFLECTION_1 = """
YOUR ANSWER HERE (4-6 sentences)
"""

print('Reflection 1:')
print(REFLECTION_1)

In [ ]:
# ══════════════════════════════════════════════════════
# REFLECTION 2 (5 pts)
# ══════════════════════════════════════════════════════
#
# QUESTION: YOLO is pre-trained on COCO (80 everyday classes).
# Industrial parts (bolts, washers, PCBs) are NOT in COCO.
#
# Explain how you would adapt a pre-trained YOLO model to
# detect custom parts. In your answer, mention:
#   (a) what transfer learning means here,
#   (b) what data you would collect and how to label it,
#   (c) why you need far fewer images than training from scratch.
#
# ─────────────────────────────────────────────────────

REFLECTION_2 = """
YOUR ANSWER HERE (4-6 sentences)
"""

print('Reflection 2:')
print(REFLECTION_2)

In [ ]:
# ══════════════════════════════════════════════════════
# REFLECTION 3 (5 pts)
# ══════════════════════════════════════════════════════
#
# QUESTION: In this homework you used both bounding-box
# detection and pixel-level instance segmentation.
#
# Describe a specific real-world scenario (not bin picking)
# where instance segmentation provides a significant
# advantage over bounding-box detection. Explain what the
# system needs to do and why exact pixel boundaries matter.
#
# ─────────────────────────────────────────────────────

REFLECTION_3 = """
YOUR ANSWER HERE (4-6 sentences)
"""

print('Reflection 3:')
print(REFLECTION_3)

---
# Submission Checklist

Run the cell below to verify your notebook is complete before submitting.

In [ ]:
# ══════════════════════════════════════════════════════
# SUBMISSION CHECKLIST
# ══════════════════════════════════════════════════════

print('SUBMISSION CHECKLIST')
print('=' * 55)
print()

# Student info
name_ok = STUDENT_NAME != 'YOUR NAME HERE'
id_ok = STUDENT_ID != 'YOUR ID HERE'
print(f'  [{chr(10003) if name_ok else chr(10007)}] Student name filled in')
print(f'  [{chr(10003) if id_ok else chr(10007)}] Student ID filled in')
print()

# Output files
files_expected = [
    ('hw_outputs/task1_3_my_image.jpg', 'Task 1.3 — your image detection'),
    ('hw_outputs/task2_3_isolated.jpg', 'Task 2.3 — isolated object'),
    ('hw_outputs/task3_2_comparison.png', 'Task 3.2 — comparison figure'),
]
print('Output files:')
for fpath, desc in files_expected:
    exists = os.path.exists(fpath)
    print(f'  [{chr(10003) if exists else chr(10007)}] {fpath} — {desc}')
print()

# Reflections
r1_ok = 'YOUR ANSWER' not in REFLECTION_1
r2_ok = 'YOUR ANSWER' not in REFLECTION_2
r3_ok = 'YOUR ANSWER' not in REFLECTION_3
print('Reflection questions:')
print(f'  [{chr(10003) if r1_ok else chr(10007)}] Reflection 1 answered')
print(f'  [{chr(10003) if r2_ok else chr(10007)}] Reflection 2 answered')
print(f'  [{chr(10003) if r3_ok else chr(10007)}] Reflection 3 answered')
print()

# Comparison
comp_ok = comparison['explanation'].strip() != '' and 'REPLACE' not in comparison['explanation']
print(f'  [{chr(10003) if comp_ok else chr(10007)}] Task 3.2 comparison written')
print()

all_ok = all([name_ok, id_ok, r1_ok, r2_ok, r3_ok, comp_ok])
files_ok = all(os.path.exists(f) for f, _ in files_expected)

if all_ok and files_ok:
    print('All checks passed! Ready to submit.')
else:
    print('Some items are incomplete. Please review above.')

---
## Grading Rubric

| Section | Task | Points | Criteria |
|---------|------|--------|----------|
| **Part 1** | 1.1 Load & detect | 10 | Model loads, detections displayed with boxes, summary printed (class, confidence, box) |
| | 1.2 Analysis | 10 | Correct count, class bar chart, confidence histogram, filtered output |
| | 1.3 Own image | 10 | Valid image with 3+ object types, detection runs, annotated image displayed and saved |
| **Part 2** | 2.1 Segmentation | 10 | Seg model loads, masks displayed, summary printed |
| | 2.2 Individual masks | 10 | Masks extracted into grid, area and coverage % computed and printed |
| | 2.3 Isolate object | 10 | Mask applied correctly, isolated on black, cropped, 3-panel displayed, saved |
| **Part 3** | 3.1 Classical pipeline | 15 | All 6 steps: preprocess, threshold, morphology, contours, visualization, statistics |
| | 3.2 Comparison | 10 | 3-panel figure saved, counts reported, thoughtful 3–5 sentence explanation |
| **Part 4** | Reflection 1 | 5 | Understands hybrid CV systems with specific example |
| | Reflection 2 | 5 | Correctly describes transfer learning, data needs, and reduced requirement |
| | Reflection 3 | 5 | Valid real-world scenario where segmentation clearly beats detection |
| | **Total** | **100** | |

**Notes:**
- Code must **run without errors** for full credit.
- Partial credit for correct approach with minor bugs.
- Reflection answers graded on **depth of understanding**, not length.
- Output images must be saved in `hw_outputs/`.

---
*CYS5550 – Weeks 7–8 Homework: Object Detection & Segmentation with YOLO — Version 1.0*